In [12]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
from s0_fun_base import XY_fec_compu
import gpflow
from s0_class_GP_IPM import Perted_IPM
from tensorflow_probability import distributions as tfd
f64 = gpflow.utilities.to_default_float
from sklearn.metrics import roc_curve, auc

In [ ]:
convert_to_constrained_values = 'ON'
truedata_style='glm'
grw_setting='sep'
target='fec'
if convert_to_constrained_values == 'ON':
    # converting unconstrained values to constrained space & calculating the likelihood values.
    true_population = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/"+truedata_style+"_mle_true_population.pkl", mode="rb"))
    df_fec = XY_fec_compu(true_population)  

    m_fec_new = gpflow.models.GPMC(data=(df_fec[0], df_fec[1]), 
                                kernel=gpflow.kernels.RBF(), likelihood=gpflow.likelihoods.Bernoulli())

    # Secondly, we add priors to the hyperparameters.
    m_fec_new.kernel.variance.prior = tfd.HalfNormal(scale=f64(100.))
    m_fec_new.kernel.lengthscales.prior = tfd.HalfNormal(scale=f64(100.))

    # We now sample from the posterior using HMC.
    hmc_helper = gpflow.optimizers.SamplingHelper(
        m_fec_new.log_posterior_density, m_fec_new.trainable_parameters
    )

    samples = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/samples_"+target+".pkl", mode="rb")) 
    constrained_samples = hmc_helper.convert_to_constrained_values(samples)
    pickle.dump(constrained_samples, open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/constrained_samples_"+target+".pkl", mode="wb"))

    nll = np.array([])
    nlp = np.array([])

    for i in range(5000):
        for var, var_samples in zip(hmc_helper.current_state, samples):
            var.assign(var_samples[i])
        
        nll = np.append(nll, np.array(m_fec_new.log_likelihood()))
        nlp = np.append(nlp, np.array(m_fec_new.log_posterior_density()))
    
    pickle.dump(nll, open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/nll_"+target+".pkl", mode="wb"))
    pickle.dump(nlp, open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/nlp_"+target+".pkl", mode="wb"))

In [13]:
truedata_style='glm'
grw_setting='sep'
target='fec'
opt_percentage = 6
rep = 100
popu_dataset = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/"+truedata_style+"_mle_true_population.pkl", mode="rb"))
models_true = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/gp_"+truedata_style+"mle_mle_models.pkl", mode="rb"))
constrained_samples = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/constrained_samples_"+target+".pkl", mode="rb"))
nlp = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/nlp_"+target+".pkl", mode="rb"))

In [15]:
IPM_pret_fec = Perted_IPM(popu_data=popu_dataset, GPmodel_true=models_true, 
                          grw_setting=grw_setting, target='m_' + target, truedata_style=truedata_style,
                          mcmc_para_sample=constrained_samples, summary='Full', nlog_post=nlp, opt_percentage=opt_percentage)

In [16]:
popu_opt_mode = 'OFF'

In [17]:
if popu_opt_mode == 'ON':
    print('\n\n\n' + 'Re-calculating summary_data for the opt MCMC samples' + '\n\n\n')
    summary_opt = IPM_pret_fec.simuVSsimu_singleModel_fun_parallel(rep=rep, random_seed=1,
                                        interested_models='Opt', evaluate_at_training=False)

    pickle.dump(summary_opt, 
                open(file = os.getcwd() + f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_opt_"+target+".pkl", mode="wb"))
    
    summary_around_opt = IPM_pret_fec.simuVSsimu_fun_parallel(rep=rep, random_seed=1, 
                                    opt_percentage=opt_percentage, interested_models='Opt', 
                                    MCMC_boolean_list='Opt', evaluate_at_training=False)
    pickle.dump(summary_around_opt, 
                open(file = os.getcwd() + f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_around_opt_"+target+".pkl", mode="wb"))
    
elif popu_opt_mode == 'OFF':
    print('\n\n\n' + 'Loading summary_data for the opt MCMC samples' + '\n\n\n')
    summary_opt = pickle.load(open(file = os.getcwd() + f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_opt_"+target+".pkl", mode="rb"))
    summary_around_opt = pickle.load(open(file = os.getcwd()+f"/true_popu/"+truedata_style+"_mle_true_population/MCMC/gp/"+grw_setting+"/summary_around_opt_"+target+".pkl", mode="rb"))




Re-calculating summary_data for the opt MCMC samples







 Parallel computing is starting 
 with repeat time 100 and random seed 1.






2024-01-11 22:52:42,432	INFO worker.py:1673 -- Started a local Ray instance.






 Parallel computing is starting 
 with repeat time 100 and random seed 1.






In [24]:
print('\n\n\n' + 'Re-calculating the top summary stats' + '\n\n\n')
num_mcmc = IPM_pret_fec.mcmc_para_sample[0].shape[0]

most_freq_summary_stats = pd.DataFrame(data=0.0, 
                                        index=range(np.sum(IPM_pret_fec.whether_around_opt_comp)), 
                                        columns=IPM_pret_fec.col_names)

for j in range(np.sum(IPM_pret_fec.whether_around_opt_comp)):
    d = summary_around_opt.loc[(0+j*rep):(rep-1+j*rep)].reset_index(drop=True).copy()
    auc0 = np.zeros(33)
    auc1 = np.zeros(33)
    for i in range(33):        
        fpr0, tpr0, _ = roc_curve(y_true=np.append(np.repeat(1, rep), np.repeat(0, rep)), 
                                y_score=np.append(summary_opt.iloc[:, i], d.iloc[:, i]), pos_label=0)
        auc0[i] = auc(fpr0, tpr0)
    
    most_freq_summary_stats.iloc[j, np.argsort(auc0)[np.sort(auc0) > 0.6]] = most_freq_summary_stats.iloc[j, np.argsort(auc0)[np.sort(auc0) > 0.6]]+ 1
    





Re-calculating the top summary stats





In [25]:
#0.6
print(np.sort(most_freq_summary_stats.sum())[-10:])
most_columns = most_freq_summary_stats.columns[np.argsort(most_freq_summary_stats.sum())[-10:]]
print(most_columns)

[ 28.  29.  29.  30.  91. 127. 150. 206. 217. 232.]
Index(['19b', '29c', '29d', '27g', '27e', '10a', '29e', '7b', '7a', '10b'], dtype='object')
